# Semana 01 · MLOps: evidencia de un experimento en MLflow

**Solución docente.** Crear dos *runs* comparables con parámetros, métricas, artefactos y riesgos. Está preparado para Databricks Free Edition: cada notebook registra en su propio experimento gestionado.

> El dataset es únicamente didáctico. No debe utilizarse para diagnóstico ni decisiones clínicas.

## 0. Dependencias

Ejecuta la siguiente celda sólo si el runtime no ofrece MLflow 3.1 o superior. Tras `%pip`, reinicia Python y continúa desde la celda de configuración.

In [ ]:
%pip install -q --upgrade "mlflow[databricks]>=3.1" scikit-learn pandas

## 1. Configuración reproducible

No usamos `set_experiment()`: en un notebook de Free Edition, MLflow crea o reutiliza automáticamente el experimento del propio notebook. El alias identifica al trabajo sin usar correos ni nombres completos.

In [ ]:
import json

import mlflow
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

STUDENT_ALIAS = "pareja-07"
RUN_LABEL = "baseline-heart"
MODEL_CONFIG = {
    "n_estimators": 80,
    "max_depth": 4,
    "min_samples_leaf": 3,
    "random_state": 42,
}

mlflow.set_tracking_uri("databricks")
print(f"MLflow {mlflow.__version__}; experimento: notebook actual")

## 2. Datos, límites y riesgos

Un registro de riesgos y una tarjeta de datos son artefactos de la ejecución. Así quedan ligados a la configuración y a las métricas que ayudaron a tomar la decisión.

In [ ]:
# Esta ruta funciona al abrir el repositorio como Databricks Git Folder.
DATASET_PATH = "../../../data/raw/heart.csv"
data = pd.read_csv(DATASET_PATH)
features = data.drop(columns="target")
target = data["target"]
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)
categorical_columns = features.select_dtypes(include=["object", "bool"]).columns.tolist()
numeric_columns = [column for column in features.columns if column not in categorical_columns]
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([("impute", SimpleImputer(strategy="median"))]), numeric_columns),
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder(handle_unknown="ignore"))]), categorical_columns),
    ]
)

dataset_card = {
    "source": "data/raw/heart.csv (caso histórico del repositorio)",
    "purpose": "Práctica didáctica de tracking; no es un sistema clínico.",
    "rows": int(features.shape[0]),
    "features": list(features.columns),
    "split": {"test_size": 0.2, "random_state": 42, "stratify": "target"},
    "known_limitations": [
        "No representa una población clínica real.",
        "No debe utilizarse para diagnóstico o tratamiento.",
    ],
}
risk_register = {
    "version": "s01",
    "risks": [
        {
            "id": "R1",
            "risk": "Uso clínico fuera del contexto docente",
            "impact": "high",
            "owner": "equipo de producto",
            "mitigation": "Supervisión humana y prohibición expresa de uso clínico.",
        },
        {
            "id": "R2",
            "risk": "Pérdida de reproducibilidad",
            "impact": "medium",
            "owner": "responsable técnico",
            "mitigation": "Semilla, parámetros, métricas y tarjeta de datos en MLflow.",
        },
        {
            "id": "R3",
            "risk": "Registro accidental de secretos o datos sensibles",
            "impact": "high",
            "owner": "todo el equipo",
            "mitigation": "Nunca registrar tokens, PII o prompts sensibles.",
        },
    ],
}

display(features.head())

## 3. Entrenar y registrar un run

El run conserva configuración, métricas y artefactos, pero no registra todavía el modelo. La promoción requiere contratos y validaciones de semanas posteriores.

In [ ]:
mlflow.sklearn.autolog(log_models=False, silent=True)

with mlflow.start_run(run_name=f"{STUDENT_ALIAS}-{RUN_LABEL}") as run:
    mlflow.set_tags(
        {
            "course": "MUIAAP-operacion-modelos",
            "course.week": "01",
            "student.alias": STUDENT_ALIAS,
            "use_case": "didactic-heart-risk-classification",
            "risk.tier": "not-for-production",
        }
    )
    mlflow.log_params(MODEL_CONFIG)
    mlflow.log_dict(dataset_card, "governance/dataset_card.json")
    mlflow.log_dict(risk_register, "governance/risk_register.json")
    mlflow.log_dict(
        {"input_example": X_test.head(3).to_dict(orient="records")},
        "evidence/input_example.json",
    )

    model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", RandomForestClassifier(**MODEL_CONFIG)),
        ]
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    metrics = {
        "test_f1": float(f1_score(y_test, predictions)),
        "test_recall": float(recall_score(y_test, predictions)),
        "test_roc_auc": float(roc_auc_score(y_test, probabilities)),
    }
    mlflow.log_metrics(metrics)
    run_id = run.info.run_id
    experiment_id = run.info.experiment_id

experiment = mlflow.get_experiment(experiment_id)
print(f"Run: {run_id}")
print(f"Experimento de notebook: {experiment.name}")
print(json.dumps(metrics, indent=2))

## 4. Comparar dos runs

Cambia una sola decisión de `MODEL_CONFIG` y `RUN_LABEL`, vuelve a ejecutar las secciones 1–3 y compara. Una métrica mejor no autoriza producción: revisa también límites de datos, riesgos y uso previsto.

In [ ]:
runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string="tags.course.week = '01'",
    order_by=["metrics.test_f1 DESC"],
    max_results=20,
)
columns_to_show = [
    "run_id",
    "tags.mlflow.runName",
    "params.max_depth",
    "params.min_samples_leaf",
    "metrics.test_f1",
    "metrics.test_recall",
    "metrics.test_roc_auc",
]
display(runs.reindex(columns=columns_to_show))

## Comprobación de salida

En **Experiments**, comprueba que hay dos runs con parámetros y métricas, y que cada uno contiene `governance/dataset_card.json` y `governance/risk_register.json`. Revisa que no haya secretos ni datos personales antes de compartir una captura o un enlace.